<a href="https://colab.research.google.com/github/builtbyC/INFO570_term_project/blob/main/sbus_sa_1.3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Cell 1 — Imports
import pandas as pd
import numpy as np
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment
import warnings
warnings.filterwarnings('ignore')

In [3]:
# Cell 2 — Inputs & File Uploads
from google.colab import files
import io
from zoneinfo import ZoneInfo

report_date_str           = input("Report date (YYYY-MM-DD, EST): ").strip()
grade_threshold           = float(input("Grade threshold (0–1, e.g. 0.85): ").strip())
missing_alert_threshold   = int(input("Action needed if missing assignments >= : ").strip())
late_alert_threshold      = int(input("Monitor if late assignments >= : ").strip())
low_score_alert_threshold = int(input("Action needed if low-score assignments >= : ").strip())

# Treat user input as EST end-of-day midnight, convert to UTC for filtering
EST = ZoneInfo("America/New_York")
report_date_est = pd.Timestamp(report_date_str, tz=EST)
report_date     = report_date_est.astimezone(ZoneInfo("UTC"))  # used for all Canvas filtering

OUTPUT_FILE = f"advisor_report_{report_date_est.strftime('%Y%m%d')}.xlsx"

print(
    f"\ndate={report_date_est.date()} EST → {report_date} UTC | "
    f"grade_threshold={grade_threshold} | "
    f"missing>={missing_alert_threshold} | late>={late_alert_threshold} | "
    f"low_score>={low_score_alert_threshold}"
)

def read_uploaded_file(uploaded_dict):
    filename = next(iter(uploaded_dict))
    file_bytes = uploaded_dict[filename]
    if filename.lower().endswith(".csv"):
        df = pd.read_csv(io.BytesIO(file_bytes))
    elif filename.lower().endswith(".xlsx"):
        df = pd.read_excel(io.BytesIO(file_bytes))
    else:
        raise ValueError(f"Unsupported file format: {filename}")
    return filename, df

print("\nUpload file 1: Summary file")
uploaded_summary = files.upload()

print("\nUpload file 2: Canvas report")
uploaded_canvas = files.upload()

print("\nUpload file 3: Cognos data")
uploaded_cognos = files.upload()

summary_filename, summary_df = read_uploaded_file(uploaded_summary)
canvas_filename,  canvas_df  = read_uploaded_file(uploaded_canvas)
cognos_filename,  cognos_df  = read_uploaded_file(uploaded_cognos)

print("\nFiles received:")
print(f"  - Summary: {summary_filename}")
print(f"  - Canvas:  {canvas_filename}")
print(f"  - Cognos:  {cognos_filename}")

Report date (YYYY-MM-DD, EST): 2026-02-23
Grade threshold (0–1, e.g. 0.85): .85
Action needed if missing assignments >= : 1
Monitor if late assignments >= : 1
Action needed if low-score assignments >= : 1

date=2026-02-23 EST → 2026-02-23 05:00:00+00:00 UTC | grade_threshold=0.85 | missing>=1 | late>=1 | low_score>=1

Upload file 1: Summary file


Saving SBUS_grades.csv to SBUS_grades.csv

Upload file 2: Canvas report


Saving canvas_grades_detailed_2026-02-23.csv to canvas_grades_detailed_2026-02-23.csv

Upload file 3: Cognos data


Saving SBUS Enrollment and Tuition.xlsx to SBUS Enrollment and Tuition.xlsx

Files received:
  - Summary: SBUS_grades.csv
  - Canvas:  canvas_grades_detailed_2026-02-23.csv
  - Cognos:  SBUS Enrollment and Tuition.xlsx


In [4]:
# Cell 3 — Configuration

OUTPUT_FILE = f"advisor_report_{report_date_est.strftime('%Y%m%d')}.xlsx"

# ── Column order ──────────────────────────────────────────────────────────────
# To reorder: move an item to a different group, or move a whole group up/down.
# Assignment columns are always appended last, auto-sorted by due date.
STATIC_COL_GROUPS = [
    ["NAME", "ID", "MSU_EMAIL"],
    ["REGISTRATION_COHORT_DESCRIPTION", "ACADEMIC_STANDING", "CUMULATIVE_GPA", "PROGRAM_GPA"],
    ["LOW_GRADE_LIST", "CLASSES_REGISTERED_CURRENT_TERM", "CURRENT_COURSE_IN_PROGRESS"],
    ["current_score", "final_score", "current_grade", "final_grade"],
    ["total_assignments", "submitted_count", "total_missing", "total_late", "total_low_score"],
    # Assignment columns follow automatically — do not add them here
]

In [5]:
# Cell 4 — Load Files

df_summary = summary_df.copy()
df_canvas  = canvas_df.copy()
df_cognos  = cognos_df.copy()

# Normalize column names for summary and canvas; leave cognos as-is
df_summary.columns = df_summary.columns.str.strip().str.lower().str.replace(' ', '_')
df_canvas.columns  = df_canvas.columns.str.strip().str.lower()

print(f"Loaded — summary: {df_summary.shape}, canvas: {df_canvas.shape}, cognos: {df_cognos.shape}")

Loaded — summary: (21289, 23), canvas: (3345, 22), cognos: (5109, 51)


In [6]:
# Cell 5 — Preprocess

# ── Summary ───────────────────────────────────────────────────────────────────
df_summary = df_summary[df_summary['enrollment_state'] == 'active'].copy()
df_summary['student_sis'] = df_summary['student_sis'].astype(str).str.strip()

# ── Canvas ────────────────────────────────────────────────────────────────────
df_canvas['sis_user_id'] = df_canvas['sis_user_id'].astype(str).str.strip()
df_canvas['assignment_due_at'] = pd.to_datetime(df_canvas['assignment_due_at'], utc=True, errors='coerce')

# Drop rows where due date could not be parsed
df_canvas = df_canvas[df_canvas['assignment_due_at'].notna()].copy()

# Include only assignments due on or before report_date
df_canvas = df_canvas[df_canvas['assignment_due_at'] <= report_date].copy()

# Drop assignments with a max possible score of 0 — placeholder/ungraded items
df_canvas = df_canvas[df_canvas['points_possible'] > 0].copy()

# Normalize late/missing to bool (Canvas exports these as TRUE/FALSE strings)
for col in ['late', 'missing']:
    df_canvas[col] = df_canvas[col].astype(str).str.upper().isin(['TRUE', '1', 'YES'])

# ── Cognos ────────────────────────────────────────────────────────────────────
# Strip M prefix from student ID to align with Canvas/Summary identifiers
df_cognos['ID'] = df_cognos['ID'].astype(str).str.strip().str.lstrip('M')

print(f"After filters — active enrollments: {len(df_summary)}, canvas rows: {len(df_canvas)}")

After filters — active enrollments: 20939, canvas rows: 1013


In [7]:
# Cell 6 — Build Per-Course DataFrames

def get_assignment_status(missing, late):
    """missing takes precedence over late."""
    if missing:
        return 'missing'
    if late:
        return 'late'
    return 'good'


def build_course_df(df_cv, df_sm):
    """
    Builds a wide-format DataFrame for a single course.
    Rows = students; columns = static fields + per-assignment status/grade pairs
    (sorted by due date, appended last).
    """
    assignments = (
        df_cv[['assignment_name', 'assignment_due_at']]
        .drop_duplicates('assignment_name')
        .sort_values('assignment_due_at')['assignment_name']
        .tolist()
    )

    records = []
    for _, srow in df_sm.iterrows():
        sid = srow['student_sis']
        sc  = df_cv[df_cv['sis_user_id'] == sid]

        total_missing   = 0
        total_late      = 0
        total_low_score = 0
        submitted_count = 0
        record = {'student_sis': sid}

        for asgn in assignments:
            arow = sc[sc['assignment_name'] == asgn]

            if arow.empty:
                status = 'missing'
                grade  = np.nan
            else:
                arow   = arow.iloc[0]
                status = get_assignment_status(arow['missing'], arow['late'])
                grade  = round(arow['score'] / arow['points_possible'] * 100, 2)

            if status == 'missing':
                total_missing += 1
            else:
                submitted_count += 1
                if status == 'late':
                    total_late += 1
                if not np.isnan(grade) and (grade / 100) < grade_threshold:
                    total_low_score += 1

            record[f'{asgn}_status'] = status
            record[f'{asgn}_grade']  = grade

        record['total_assignments'] = len(assignments)
        record['submitted_count']   = submitted_count
        record['total_missing']     = total_missing
        record['total_late']        = total_late
        record['total_low_score']   = total_low_score

        records.append(record)

    df_wide = pd.DataFrame(records)

    score_cols = ['student_sis', 'current_score', 'final_score', 'current_grade', 'final_grade']
    available  = [c for c in score_cols if c in df_sm.columns]
    df_wide    = df_wide.merge(df_sm[available], on='student_sis', how='left')

    return df_wide, assignments


# ── Cognos columns carried into each sheet (no PERSONAL_EMAIL or PHONE_NUMBER)
COGNOS_KEEP = [
    'ID', 'NAME', 'MSU_EMAIL',
    'REGISTRATION_COHORT_DESCRIPTION', 'ACADEMIC_STANDING',
    'CUMULATIVE_GPA', 'PROGRAM_GPA',
    'LOW_GRADE_LIST', 'CLASSES_REGISTERED_CURRENT_TERM', 'CURRENT_COURSE_IN_PROGRESS',
]
df_cognos_slim = df_cognos[[c for c in COGNOS_KEEP if c in df_cognos.columns]].copy()


def apply_col_order(df, assignment_cols):
    """Order columns per STATIC_COL_GROUPS; assignment columns always last."""
    static = [c for group in STATIC_COL_GROUPS for c in group if c in df.columns]
    return df[[c for c in static + assignment_cols if c in df.columns]]


# ── Build one DataFrame per course ───────────────────────────────────────────
course_sheets = {}

for course_code in df_canvas['course_name'].unique():
    df_cv = df_canvas[df_canvas['course_name'] == course_code].copy()
    df_sm = df_summary[df_summary['course'] == course_code].copy()

    if df_sm.empty:
        continue

    descriptive_name = df_cv['course_section'].iloc[0]

    df_wide, assignments = build_course_df(df_cv, df_sm)

    asgn_cols = [
        col
        for asgn in assignments
        for col in (f'{asgn}_status', f'{asgn}_grade')
    ]

    df_wide = df_wide.merge(df_cognos_slim, left_on='student_sis', right_on='ID', how='left')
    course_sheets[descriptive_name] = apply_col_order(df_wide, asgn_cols)

    print(f"  {descriptive_name}: {len(df_wide)} students, {len(assignments)} assignments")

  MKTG562_01SP26: 34 students, 6 assignments
  INFO584_91SP26: 21 students, 26 assignments
  INFO584_92SP26: 29 students, 26 assignments
  MGMT562_70SP26: 148 students, 33 assignments
  INFO591_91SP26: 8 students, 1 assignments
  INFO588_91SP26: 57 students, 5 assignments


In [8]:
# Cell 7 — Build Summary Sheet
# One row per student; per-course: current_score, final_score, total_missing, total_late, total_low_score

summary_parts = []

for course_name, df_course in course_sheets.items():
    id_col = 'ID' if 'ID' in df_course.columns else 'student_sis'
    keep   = [c for c in [
                  id_col, 'NAME',
                  'current_score', 'final_score',
                  'total_missing', 'total_late', 'total_low_score'
              ] if c in df_course.columns]
    temp   = df_course[keep].copy()

    prefix = course_name[:20].strip()
    temp   = temp.rename(columns={
        'current_score':   f'{prefix}_current_score',
        'final_score':     f'{prefix}_final_score',
        'total_missing':   f'{prefix}_missing',
        'total_late':      f'{prefix}_late',
        'total_low_score': f'{prefix}_low_score',
    })
    summary_parts.append(temp)

df_summary_out = summary_parts[0]
id_col = 'ID' if 'ID' in df_summary_out.columns else 'student_sis'

for part in summary_parts[1:]:
    df_summary_out = df_summary_out.merge(part, on=[id_col, 'NAME'], how='outer')

print(f"Summary sheet: {df_summary_out.shape}")

Summary sheet: (274, 32)


In [9]:
# Cell 8 — Write Excel Output

def safe_sheet_name(name, used, max_len=31):
    """Truncate to Excel's 31-char sheet name limit and ensure uniqueness."""
    base = name[:max_len]
    if base not in used:
        return base
    for i in range(1, 100):
        candidate = f"{name[:max_len - 3]}_{i}"
        if candidate not in used:
            return candidate
    raise ValueError(f"Could not generate a unique sheet name for: {name}")


with pd.ExcelWriter(OUTPUT_FILE, engine='openpyxl') as writer:
    df_summary_out.to_excel(writer, sheet_name='Summary', index=False)

    used_names = {'Summary'}
    for course_name, df_course in course_sheets.items():
        sheet_name = safe_sheet_name(course_name, used_names)
        used_names.add(sheet_name)
        df_course.to_excel(writer, sheet_name=sheet_name, index=False)

print(f"Report written → {OUTPUT_FILE}")

# Download the file
files.download(OUTPUT_FILE)

Report written → advisor_report_20260223.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>